# Testing baseline with RL schedule sensitivity

Replicates the **GRU / 30-minute** configuration in `results/testing`, using
12 OhioT1DM patients (2018 + 2020) and seeds **43, 44**.

| Arm | RL training | Shared TL reference |
|---|---|---|
| Baseline | Up to 200 epochs, patience 5 | Up to 10 epochs at 0.0003 + up to 10 at 0.00005; patience 5 per stage |
| Longer patience | Up to 200 epochs, patience 15 | Same fitted TL models |
| Fixed budget | 20 epochs, no early stopping | Same fitted TL models |

All arms use GRU with 128 hidden units, two layers, dropout 0.2, batch size 16,
60 minutes of input, and six features (glucose, basal, bolus, carbs, hour sine/cosine).
Training uses Adam, MSE, weight decay 0.00001, gradient clipping 1.0, and the
same plateau scheduler. RL is single-stage at initial learning rate 0.0003.
TL resets Adam between stages and uses patience 5 independently of RL patience.
Both regimes select the best validation checkpoint. Metrics use the final forecast step.

CPU is the default to match the saved runs; set `DEVICE_CFG = "cuda"` and select
a GPU runtime for acceleration. This matches the saved hyperparameters, not the
exact historical fitted models: the current runner and paired per-patient RNG
scheme are retained. Compare schedule arms within this run.

Before running on Colab, push this notebook and the updated
`benchmark/configs/config_manager.py`, `benchmark/experiments/configured.py`, and
`benchmark/models/base_model.py` to the selected branch. Completed arms are
validated and saved to Drive; interrupted arms rerun in full.


## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"      # must contain 2018/ and 2020/
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results"    # results are mirrored here
REPO_DIR      = "/content/BG-forecasting-cui-patience"

SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"

SEEDS      = [43, 44]
DEVICE_CFG = "cpu"         # matches results/testing; use "cuda" for a GPU rerun
# ---------------------------------------------------------------------------

import os, pathlib

# The Drive layout is run_on_colab.ipynb's, so results are fetched from and
# written to the same places as the rest of the project: runs under
# experiments/, derived tables under analysis/ (which mirrors results/analysis),
# training output under logs/.
DRIVE_EXPERIMENTS = f"{DRIVE_RESULTS}/experiments"
DRIVE_ANALYSIS    = f"{DRIVE_RESULTS}/analysis/sensitivity"
DRIVE_LOGS        = f"{DRIVE_RESULTS}/logs"
for d in (DRIVE_EXPERIMENTS, DRIVE_ANALYSIS, DRIVE_LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Drive ready.")
print("  runs   ->", DRIVE_EXPERIMENTS)
print("  tables ->", DRIVE_ANALYSIS)
print("  logs   ->", DRIVE_LOGS)
NOTEBOOK_CODE_SHA256 = 'f1acdcc7e86e4d2553401e6e3166d685630f49709a05a2027edbad2a13fb9e4a'


## 2 · Get the code

In [ ]:
import shutil, subprocess, sys, pathlib

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            raise RuntimeError("git pull --ff-only failed; resolve the checkout before running.\n"
                  + (pull.stderr or pull.stdout).strip() + "\n")
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

BASE_CONFIG = pathlib.Path(REPO_DIR) / "configs" / "full_gru_30min.yaml"
if not BASE_CONFIG.is_file():
    raise SystemExit(f"{BASE_CONFIG} missing; the arms are derived from it.")
print("Base config:", BASE_CONFIG)

## 3 · Stage the OhioT1DM data

In [ ]:
import shutil, pathlib, hashlib

def sha(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()

COHORT = {"2018": [559, 563, 570, 575, 588, 591],
          "2020": [540, 544, 552, 567, 584, 596]}

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found. Upload your OhioT1DM copy there first.")

copied = 0
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        (dst / release / mode).mkdir(parents=True, exist_ok=True)
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            target, source = dst / release / mode / name, src / release / mode / name
            if not source.is_file():
                raise FileNotFoundError(source)
            if target.is_file() and sha(target) == sha(source):
                continue
            shutil.copy2(source, target)
            copied += 1
            assert sha(target) == sha(source)
print(f"{copied} file(s) copied from Drive.")

missing = []
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            if not (dst / release / mode / name).is_file():
                where = "absent from Drive too" if not (src / release / mode / name).is_file() else "copy failed"
                missing.append(f"{release}/{mode}/{name} ({where})")
if missing:
    raise SystemExit("Missing data files:\n  " + "\n  ".join(missing))
print("All 24 XML files staged.")

## 4 · Write the three configs

The setup overrides the base grid with the GRU/30-minute testing configuration:
six features, seeds 43/44, `regular_schedule: single_stage`, weight decay 0.00001,
and gradient clipping 1.0. `transfer_early_stopping_patience: 5` holds the TL
stopping policy fixed while RL patience varies. Each TL stage has a 10-epoch cap.
The base grid files are not rewritten. Generated configs are schema-validated.


In [ ]:
import yaml, copy, pathlib, json, hashlib, platform
import numpy as np, pandas as pd

base = yaml.safe_load(BASE_CONFIG.read_text())

pre, train = base["preprocessing"], base["training"]
# Pin the saved testing baseline explicitly; the base grid currently uses F=4.
base['data'].update(dataset='ohiot1dm', version='both', patients=sorted(sum(COHORT.values(), [])),
                    train_ratio=0.9, validation_ratio=0.1)
pre.update(unimodal=False, include_feature_engineering=True, prediction_horizon=6,
           window_size=12, sampling_rate=5, normalization='standardize')
base['model'].update(type='gru', architecture=dict(
    hidden_size=128, num_layers=2, dropout=0.2, batch_first=True))
train.update(regular_schedule='single_stage', learning_rate=0.0003,
             finetune_learning_rate=0.00005, pretrain_epochs=10, finetune_epochs=10,
             batch_size=16, transfer_early_stopping_patience=5, seeds=SEEDS,
             weight_decay=0.00001, grad_clip_norm=1.0)

ARMS = {
    "sens_gru_30min_arm0_patience5": dict(
        mode="both", epochs=200, early_stopping_patience=5,
        note="Testing-baseline TL held fixed; supplies the patience-5 RL reference"),
    "sens_gru_30min_armA_patience15": dict(
        mode="regular", epochs=200, early_stopping_patience=15,
        note="R3 #3 directly: was patience 5 cutting RL short?"),
    "sens_gru_30min_armB_fixed20": dict(
        mode="regular", epochs=20, early_stopping_patience=20,
        note="20 target epochs; best-validation checkpoint; unequal compute to TL"),
}

# One device for all three arms, decided before anything is written, so the
# comparison between rows cannot straddle two devices.
effective_device = DEVICE_CFG or base["training"]["device"]
config_device = base["training"]["device"]
if DEVICE_CFG is not None and DEVICE_CFG != config_device:
    print(f"NOTE: {BASE_CONFIG.name} says device: {config_device}; the arms are pinned to "
          f"{DEVICE_CFG}. Only the arms are affected -- the base config is not rewritten.")
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
if effective_device == "cuda" and not torch.cuda.is_available():
    raise SystemExit(
        "The arms are configured for cuda but this runtime has no GPU.\n"
        "Runtime > Change runtime type > T4 GPU, or set DEVICE_CFG = 'cpu' above "
        "(slower; duration has not been measured).")
if effective_device == "auto":
    print("NOTE: device 'auto' resolves per runtime (cuda -> mps -> cpu); "
          "pin DEVICE_CFG if the arms may run in different sessions.")

written = {}
for name, arm in ARMS.items():
    cfg = copy.deepcopy(base)
    cfg["experiment"]["name"] = name
    cfg["experiment"]["description"] = f"schedule sensitivity (R3 #3): {arm['note']}"
    cfg["training"]["mode"] = arm["mode"]
    cfg["training"]["epochs"] = arm["epochs"]
    cfg["training"]["early_stopping_patience"] = arm["early_stopping_patience"]
    cfg["training"]["device"] = effective_device
    # Not read by this check; both cost time and disk.
    cfg["output"]["save_predictions"] = False
    cfg["output"]["generate_plots"] = False
    path = pathlib.Path(REPO_DIR) / "configs" / f"{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    written[name] = path
    print(f"{name:32s} mode={arm['mode']:8s} epochs={arm['epochs']:3d} "
          f"patience={arm['early_stopping_patience']:3d}  -> {path.name}")

print("\ndevice:", effective_device,
      "(inherited from the base config)" if DEVICE_CFG is None else "(pinned by DEVICE_CFG)")
print("Pinned: GRU 128x2, F=6, dropout 0.2, batch 16, weight decay 1e-5, clip 1.0; "
      "RL LR 0.0003; TL LR 0.0003 -> 0.00005, patience 5 per stage")
# Fingerprint every relevant source and raw input, not only an experiment name.
# Resolve defaults through the repository schema before saving/comparing configs.
from benchmark.configs import load_config

source_hashes = {str(f.relative_to(REPO_DIR)): sha(f)
                 for f in sorted(pathlib.Path(REPO_DIR).joinpath('benchmark').rglob('*.py'))}
data_hashes = {str(f.relative_to(dst)): sha(f) for f in sorted(dst.rglob('*.xml'))}
protocol = dict(base=base, arms=ARMS, seeds=SEEDS, device=effective_device,
                sources=source_hashes, inputs=data_hashes,
                torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,
                python=platform.python_version(), cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                notebook_code_sha256=NOTEBOOK_CODE_SHA256,
                protocol_version=4, patient_seed_rule='seed * 100000 + patient_id')
RUN_ID = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
RUN_LOCAL = pathlib.Path(REPO_DIR) / 'results' / 'cui_patience_sensitivity' / RUN_ID
RUN_DRIVE = pathlib.Path(DRIVE_RESULTS) / 'cui_patience_sensitivity' / RUN_ID
for folder in [RUN_LOCAL, RUN_DRIVE]:
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'protocol.json').write_text(json.dumps(protocol, indent=2))
DRIVE_EXPERIMENTS = str(RUN_DRIVE / 'experiments')
DRIVE_ANALYSIS = str(RUN_DRIVE / 'analysis')
DRIVE_LOGS = str(RUN_DRIVE / 'logs')
for folder in [DRIVE_EXPERIMENTS, DRIVE_ANALYSIS, DRIVE_LOGS]:
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)
for name, path in written.items():
    cfg = yaml.safe_load(path.read_text())
    cfg['output']['directory'] = str(RUN_LOCAL / 'experiments')
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
EXPECTED_CONFIGS = {name: load_config(path).to_dict() for name, path in written.items()}
assert load_config(BASE_CONFIG).data.patient_ids() == sorted(sum(COHORT.values(), [])), 'Unexpected base cohort'
print('Isolated run:', RUN_DRIVE)


## 5 · Train and restore completed arms

The three arms run sequentially. Completed arms are validated and mirrored to Drive. An interrupted arm is rerun in full; name-only or partial aggregates are never accepted. Keep the configuration and repository revision fixed across sessions to resume this fingerprint.

Patient RNGs are reset before loader/model construction. The analysis additionally checks that all RL arms share their initial validation-loss trajectory. A mismatch stops analysis rather than reporting a confounded schedule comparison.


In [ ]:
import subprocess, sys, os, pathlib, shutil, time, collections, yaml, json

EXP_DIR = RUN_LOCAL / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

# A fresh Colab VM has no local results. Restore from Drive first, or finished
# arms get retrained. Only this notebook's arms are copied: the same folder holds
# the publication grid, and pulling all of it back would move gigabytes of
# prediction CSVs that nothing here reads.
restored = 0
for saved in sorted(pathlib.Path(DRIVE_EXPERIMENTS).glob("experiment_*")):
    resolved = saved / "resolved_config.yaml"
    if not resolved.is_file():
        continue
    try:
        name = yaml.safe_load(resolved.read_text()).get("experiment", {}).get("name")
    except Exception:
        continue
    if name in ARMS:
        shutil.copytree(saved, EXP_DIR / saved.name, dirs_exist_ok=True)
        restored += 1
print(f"Restored {restored} arm run(s) from {DRIVE_EXPERIMENTS}.\n")


EXPECTED_PATIENTS = sorted(sum(COHORT.values(), []))

def checked_metrics(parent, mode, seed):
    path = parent / mode / f'seed_{seed}' / 'metrics.json'
    entries = json.loads(path.read_text())
    if set(map(int, entries)) != set(EXPECTED_PATIENTS):
        raise ValueError(f'Incomplete or unexpected cohort: {path}')
    for entry in entries.values():
        if not np.isfinite(float(entry['mae'])) or float(entry['mae']) < 0:
            raise ValueError(f'Invalid MAE: {path}')
        history = entry['training_history']
        if mode == 'transfer':
            stages = [history.get(key, {}).get('epochs_completed')
                      for key in ('pretrain_history', 'finetune_history')]
            if any(not isinstance(count, int) or not 1 <= count <= 10 for count in stages):
                raise ValueError(f'TL stage history missing or outside the 10-epoch cap: {path}')
            if history['epochs_completed'] != sum(stages):
                raise ValueError(f'TL total epochs differ from stage histories: {path}')
        if history['epochs_completed'] < 1:
            raise ValueError(f'Missing training history: {path}')
    return entries


def find_arm(name):
    done = []
    expected = EXPECTED_CONFIGS[name]
    for resolved in EXP_DIR.glob('*/resolved_config.yaml'):
        if not (resolved.parent / 'aggregate_metrics.json').is_file():
            continue
        try:
            cfg = yaml.safe_load(resolved.read_text())
            # Compare explicit settings; resolved config can add schema defaults.
            def contains(actual, required):
                return all(k in actual and (contains(actual[k], v) if isinstance(v, dict)
                           else actual[k] == v) for k, v in required.items())
            if not contains(cfg, expected):
                continue
            for mode in (['regular', 'transfer'] if ARMS[name]['mode'] == 'both' else ['regular']):
                for seed in SEEDS:
                    checked_metrics(resolved.parent, mode, seed)
            done.append(resolved.parent)
        except (ValueError, KeyError, OSError, TypeError):
            continue
    return max(done, key=lambda p: p.stat().st_mtime) if done else None


def mirror(parent):
    target = pathlib.Path(DRIVE_EXPERIMENTS) / parent.name
    shutil.copytree(parent, target, dirs_exist_ok=True)
    return target

# Isolated process-local wrapper; the repository and publication runner are not edited.
LAUNCH = """
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import sys, runpy, torch
import benchmark.experiments.configured as configured
original = configured._make_loaders

def paired_loaders(config, mode, seed, patient_id, frames):
    configured._seed_everything(seed * 100000 + patient_id)
    return original(config, mode, seed, patient_id, frames)

configured._make_loaders = paired_loaders
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
sys.argv = ['benchmark.cli', 'run', '--config', sys.argv[1]]
runpy.run_module('benchmark.cli', run_name='__main__')
"""


KEEP = ("Patient", "[INFO]", "Early stopping", "Completed", "Parent experiment",
        "ERROR", "Traceback", "Error")


def run_arm(name):
    existing = find_arm(name)
    if existing is not None:
        mirror(existing)
        print(f"{name}: already complete at {existing.name} — skipping")
        return existing
    log_path = pathlib.Path(DRIVE_LOGS) / f"{name}.log"
    print(f"{name}: starting (log -> {log_path})")
    t0 = time.time()
    tail = collections.deque(maxlen=40)
    with open(log_path, "w") as log:
        proc = subprocess.Popen([sys.executable, "-u", "-c", LAUNCH, str(written[name])],
                                cwd=REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            log.flush()
            tail.append(line.rstrip())
            if any(k in line for k in KEEP):
                print("   ", line.rstrip(), flush=True)
        rc = proc.wait()
    if rc != 0:
        print(f"    --- last {len(tail)} line(s) ---")
        for line in tail:
            print("    " + line)
        raise SystemExit(f"{name} failed (exit {rc}); full log at {log_path}")
    parent = find_arm(name)
    if parent is None:
        raise SystemExit(f"{name} reported success but wrote no aggregate_metrics.json")
    print(f"{name}: done in {(time.time()-t0)/60:.1f} min -> {mirror(parent)}")
    return parent


arm_dirs = {}
for name in ARMS:                      # arm 0 first: it trains both regimes
    arm_dirs[name] = run_arm(name)
print("\nArms:", {k: v.name for k, v in arm_dirs.items()})

## 6 · The three-row table

Built from each arm's per-seed `metrics.json`, using the same estimator the
paper uses: patients are the inferential unit, each patient's MAE is averaged
over seeds before testing, and the 95% intervals come from 20,000 resamples that
draw patients and seeds jointly, with the **same draws applied to every row** so
the rows are comparable and the `change vs arm 0` column is a paired quantity.

`benefit` is RL − TL at the advertised horizon. TL is arm 0's throughout.

In [ ]:
import json, pathlib, numpy as np, pandas as pd
from scipy.stats import wilcoxon

REPLICATES = 20000
RESAMPLING_SEED = 42


def mae_matrix(parent, mode):
    """(n_patients, n_seeds) horizon-step MAE, patients in a fixed order."""
    per_seed = {}
    for seed in SEEDS:
        path = parent / mode / f"seed_{seed}" / "metrics.json"
        if not path.is_file():
            raise SystemExit(f"Missing {path}")
        per_seed[seed] = {int(k): float(v["mae"]) for k, v in checked_metrics(parent, mode, seed).items()}
    patients = EXPECTED_PATIENTS
    return patients, np.array([[per_seed[s][p] for s in SEEDS] for p in patients])


patients, tl = mae_matrix(arm_dirs["sens_gru_30min_arm0_patience5"], "transfer")
rl = {}
for name, parent in arm_dirs.items():
    arm_patients, matrix = mae_matrix(parent, "regular")
    if arm_patients != patients:
        raise SystemExit(f"{name} has a different patient set: {arm_patients}")
    rl[name] = matrix
print(f"{len(patients)} patients x {len(SEEDS)} seeds; TL from arm 0.\n")

# Same schedule prefix must yield the same trajectory, including later patients.
reference_parent = arm_dirs['sens_gru_30min_arm0_patience5']
for seed in SEEDS:
    baseline = checked_metrics(reference_parent, 'regular', seed)
    for name, parent in arm_dirs.items():
        other = checked_metrics(parent, 'regular', seed)
        for pid in baseline:
            a = baseline[pid]['training_history']['val_losses']
            b = other[pid]['training_history']['val_losses']
            length = min(len(a), len(b))
            if not length or not np.allclose(a[:length], b[:length], rtol=1e-5, atol=1e-7):
                raise ValueError(f'RL trajectories not paired: {name}, patient {pid}, seed {seed}')

rng = np.random.default_rng(RESAMPLING_SEED)
n, k = len(patients), len(SEEDS)
pi = rng.integers(0, n, size=(REPLICATES, n))
si = rng.integers(0, k, size=(REPLICATES, k))


def resample(matrix):
    return matrix[pi[:, :, None], si[:, None, :]].mean(axis=(1, 2))


reference = resample(rl["sens_gru_30min_arm0_patience5"] - tl)
rows = []
for name, arm in ARMS.items():
    diff = rl[name] - tl
    drawn_diff, drawn_rl = resample(diff), resample(rl[name])
    percent = 100 * drawn_diff / drawn_rl
    per_patient = diff.mean(axis=1)
    change = drawn_diff - reference
    rows.append(dict(
        arm=name.replace("sens_gru_30min_", ""),
        rl_schedule=f"{arm['epochs']}ep/pat{arm['early_stopping_patience']}",
        rl_mae=rl[name].mean(), tl_mae=tl.mean(),
        benefit=diff.mean(),
        ci_low=np.quantile(drawn_diff, .025), ci_high=np.quantile(drawn_diff, .975),
        benefit_pct=100 * diff.mean() / rl[name].mean(),
        pct_low=np.quantile(percent, .025), pct_high=np.quantile(percent, .975),
        improved=int((per_patient > 0).sum()), n_patients=n,
        wilcoxon_p=1.0 if np.allclose(per_patient, 0) else float(wilcoxon(per_patient).pvalue),
        change_vs_arm0=float((diff - (rl["sens_gru_30min_arm0_patience5"] - tl)).mean()),
        change_low=np.quantile(change, .025), change_high=np.quantile(change, .975)))

table = pd.DataFrame(rows)
pvals = table.wilcoxon_p.to_numpy()
order = np.argsort(pvals)
q = np.empty(len(pvals))
q[order] = np.minimum.accumulate((pvals[order] * len(pvals) / np.arange(1, len(pvals)+1))[::-1])[::-1]
table['wilcoxon_bh_q'] = np.minimum(q, 1.)
out = RUN_LOCAL / "analysis" / "sensitivity_summary.csv"
out.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(out, index=False)
import shutil as _sh; _sh.copy2(out, pathlib.Path(DRIVE_ANALYSIS) / out.name)

pd.set_option("display.width", 200)
print(table.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print("\nTesting hyperparameters with paired RNGs: compare the three arms within this rerun.")
print("Saved:", out)

## 7 · Stopping diagnostics

Report completed epochs, best-validation epochs and the fraction hitting each cap. Similar stopping epochs alone do not establish that a schedule is adequate; read them alongside the paired effect intervals. The fixed-20 arm must complete exactly 20 epochs while retaining validation-based checkpoint selection.


In [ ]:
import json, numpy as np, pandas as pd, pathlib

rows = []
for name, parent in arm_dirs.items():
    cap = ARMS[name]["epochs"]
    for mode in ("regular", "transfer"):
        if not (parent / mode).is_dir():
            continue
        epochs, pretrain, finetune, best_epochs = [], [], [], []
        for seed in SEEDS:
            metrics = checked_metrics(parent, mode, seed)
            for entry in metrics.values():
                history = entry["training_history"]
                epochs.append(history["epochs_completed"])
                if mode == 'regular':
                    losses = history['val_losses']
                    if len(losses) != history['epochs_completed']:
                        raise ValueError('Validation history length differs from completed epochs')
                    best_epochs.append(int(np.argmin(losses)) + 1)
                    if name.endswith('fixed20') and history['epochs_completed'] != 20:
                        raise ValueError('Fixed-20 arm did not complete 20 epochs')
                if "pretrain_history" in history:
                    pretrain.append(history["pretrain_history"]["epochs_completed"])
                    finetune.append(history["finetune_history"]["epochs_completed"])
        epochs = np.array(epochs)
        row = dict(arm=name.replace("sens_gru_30min_", ""), mode=mode,
                   patience=ARMS[name]["early_stopping_patience"] if mode == "regular" else 5,
                   cap=cap if mode == "regular" else 20,
                   mean_epochs=epochs.mean(), median_epochs=float(np.median(epochs)),
                   min_epochs=int(epochs.min()), max_epochs=int(epochs.max()),
                   n_fits=len(epochs))
        if mode == "regular":
            row["pct_hitting_cap"] = 100 * float((epochs >= cap).mean())
            row["mean_best_epoch"] = float(np.mean(best_epochs))
        else:
            row["pct_hitting_cap"] = np.nan
            row["mean_pretrain_epochs"] = float(np.mean(pretrain))
            row["mean_finetune_epochs"] = float(np.mean(finetune))
        rows.append(row)

epoch_table = pd.DataFrame(rows)
out = RUN_LOCAL / "analysis" / "sensitivity_epochs.csv"
out.parent.mkdir(parents=True, exist_ok=True)
epoch_table.to_csv(out, index=False)
import shutil as _sh; _sh.copy2(out, pathlib.Path(DRIVE_ANALYSIS) / out.name)
print(epoch_table.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))
print("\nSaved:", out)

## 8 · Optional repository estimator cross-check

This checks arm 0 only. Compare point estimates after accounting for the sign convention (this notebook reports RL−TL). Confidence intervals can differ if the repository estimator uses different resampling conventions; agreement is not assumed. Missing feature inputs skip this optional check.


In [ ]:
import subprocess, sys, pathlib, pandas as pd

features = pathlib.Path(REPO_DIR) / "results/analysis/dataset/dataset_signal_features.csv"
drive_features = pathlib.Path(DRIVE_RESULTS) / "analysis" / "dataset" / "dataset_signal_features.csv"
if not features.is_file() and drive_features.is_file():
    features.parent.mkdir(parents=True, exist_ok=True)
    import shutil as _sh; _sh.copy2(drive_features, features)

if not features.is_file():
    print("dataset_signal_features.csv not available — skipping the cross-check.\n"
          "Section 6 stands on its own; this cell only corroborates it.")
else:
    parent = arm_dirs["sens_gru_30min_arm0_patience5"]
    outdir = RUN_LOCAL / "analysis" / "arm0_repo_estimator"
    proc = subprocess.run(
        [sys.executable, "RUN/experiments/run_shift_analysis.py",
         "--configured-aggregate", str(parent / "aggregate_metrics.json"),
         "--mode", "transfer", "--compare-transfer-benefit",
         "--model", "GRU", "--data-root", "data",
         "--bootstrap-count", "20000", "--resampling-seed", "42",
         "--output-dir", str(outdir)],
        cwd=REPO_DIR, capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print("FAILED:\n", proc.stderr[-3000:])
    else:
        bootstrap = outdir / "transfer_benefit_effect_bootstrap.csv"
        if bootstrap.is_file():
            print(pd.read_csv(bootstrap).to_string(index=False))
        import shutil as _sh
        _sh.copytree(outdir, pathlib.Path(DRIVE_ANALYSIS) / outdir.name, dirs_exist_ok=True)

## 9 · Interpretation and saved results

Read `change_vs_arm0` and its paired 95% interval. A negative change means the estimated transfer advantage shrinks under that RL schedule. Report absolute errors and the size and uncertainty of the change for every arm.

An interval containing zero does **not** demonstrate equivalence or close the reviewer objection. A narrow interval excluding a scientifically meaningful reduction would provide stronger robustness evidence; specify such a margin before examining results if making an equivalence claim. A wide interval leaves sensitivity unresolved. Similar stopping epochs alone are insufficient.

The three benefit tests have BH-adjusted q-values. Change intervals are exploratory and pointwise, not simultaneous. Inference is limited to this architecture/horizon and the same 12 patients, with overlapping source cohorts. Do not add horizons only because the first result is borderline.

Outputs are isolated at `<DRIVE_RESULTS>/cui_patience_sensitivity/<fingerprint>/`:

- `protocol.json`: code/data/config/environment provenance.
- `experiments/`: completed arm runs, all seed/patient metrics and histories.
- `analysis/sensitivity_summary.csv`: benefits, paired changes and uncertainty.
- `analysis/sensitivity_epochs.csv`: completed and best-validation epochs.
- `logs/`: training progress.

This follow-up can support a short schedule-sensitivity paragraph in the camera-ready paper once run. It cannot alone attribute earlier numerical revisions to seeds, corrected aggregation, or stopping policy.

The shared TL reference uses up to 10+10 epochs with patience 5 per stage,
matching the testing baseline. RL changes only its stopping schedule across rows.
The notebook keeps GRU/30 minutes as its scope; it does not rerun the full
three-model, four-horizon testing grid. Two seeds limit estimates of training variability.
